# Experiment 3 - Single-Layer Feed-Forward Network with Six Optimizers

**Aim:** Train a single-layer feed-forward neural network for binary
classification and compare Stochastic Gradient Descent, mini-batch gradient
descent, Momentum, Nesterov, Adagrad, and Adam.

**Theory in brief:** A single sigmoid neuron computes
`p = sigmoid(w1*x1 + w2*x2 + b)` and is trained by minimizing binary
cross-entropy. The six optimizers differ in how they use gradient history:

- **SGD:** one sample per update; noisy but frequent updates.
- **Mini-batch SGD:** averages the gradient over a small batch for a more
  stable step.
- **Momentum:** accumulates a velocity term to smooth oscillations.
- **Nesterov:** evaluates the gradient at the anticipated next position.
- **Adagrad:** scales each parameter by its accumulated squared gradient.
- **Adam:** combines first- and second-moment estimates with bias correction.

**Dataset:** A synthetic, linearly separable two-dimensional dataset generated
in code with NumPy. It is the same construction as the lab sheet:
`y = 1` when `x0 + 2*x1 + 0.5 > 0`. No local file or download is required.

**Expected result (original lab record):** all six optimizers reach about
99.75% accuracy, with losses between 0.01 and 0.05.

**Runtime:** about 30-60 seconds on CPU.

## 1. Environment setup

In [ ]:
import os

os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")
os.environ["PYTHONHASHSEED"] = "42"

import random
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split

SEED = 42
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

print(f"TensorFlow : {tf.__version__}")
print(f"NumPy      : {np.__version__}")
print(f"GPU devices: {tf.config.list_physical_devices('GPU')}")

## 2. Generate the linearly separable dataset

The decision boundary is the line `x0 + 2*x1 + 0.5 = 0`. Points above the line
are class 1; points below it are class 0. The classes are not perfectly
balanced, so the train/test split is stratified.

In [ ]:
N = 400
rng = np.random.default_rng(SEED)

X = rng.normal(size=(N, 2)).astype(np.float32)
logits = X[:, 0] + 2.0 * X[:, 1] + 0.5
y = (logits > 0.0).astype(np.float32).reshape(-1, 1)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=SEED,
    stratify=y.ravel(),
)

print("Total samples:", len(X))
print("Training samples:", len(X_train))
print("Test samples:", len(X_test))
print("Training class counts:", np.bincount(y_train.ravel().astype(int)).tolist())
print("Test class counts:", np.bincount(y_test.ravel().astype(int)).tolist())

assert X.shape == (N, 2)
assert y.shape == (N, 1)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(
    X_train[y_train.ravel() == 0, 0],
    X_train[y_train.ravel() == 0, 1],
    label="Class 0 (train)",
    alpha=0.7,
    edgecolor="k",
)
ax.scatter(
    X_train[y_train.ravel() == 1, 0],
    X_train[y_train.ravel() == 1, 1],
    label="Class 1 (train)",
    alpha=0.7,
    edgecolor="k",
)
line_x = np.linspace(X[:, 0].min() - 0.5, X[:, 0].max() + 0.5, 200)
line_y = -(line_x + 0.5) / 2.0
ax.plot(line_x, line_y, "k--", label="True boundary")
ax.set_xlabel("x0")
ax.set_ylabel("x1")
ax.set_title("Synthetic linearly separable data")
ax.legend()
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "exp03_dataset.png", dpi=150, bbox_inches="tight")
plt.show()

## 3. Model and optimizer configurations

Every optimizer trains a fresh copy of the same architecture: one Dense
sigmoid unit with two inputs. The random seed is reset before each model so
that all six runs start from the same initial weights.

In [ ]:
EPOCHS = 200

OPTIMIZER_SPECS = {
    "SGD (batch 1)": {
        "factory": lambda: tf.keras.optimizers.SGD(learning_rate=0.5),
        "batch_size": 1,
    },
    "Mini-batch SGD (32)": {
        "factory": lambda: tf.keras.optimizers.SGD(learning_rate=0.5),
        "batch_size": 32,
    },
    "Momentum (32)": {
        "factory": lambda: tf.keras.optimizers.SGD(learning_rate=0.1, momentum=0.9),
        "batch_size": 32,
    },
    "Nesterov (32)": {
        "factory": lambda: tf.keras.optimizers.SGD(
            learning_rate=0.1,
            momentum=0.9,
            nesterov=True,
        ),
        "batch_size": 32,
    },
    "Adagrad (32)": {
        "factory": lambda: tf.keras.optimizers.Adagrad(learning_rate=0.5),
        "batch_size": 32,
    },
    "Adam (32)": {
        "factory": lambda: tf.keras.optimizers.Adam(learning_rate=0.05),
        "batch_size": 32,
    },
}


def build_single_layer_model(optimizer, seed=SEED):
    tf.keras.utils.set_random_seed(seed)
    model = tf.keras.Sequential(
        [
            tf.keras.layers.Input(shape=(2,), name="features"),
            tf.keras.layers.Dense(1, activation="sigmoid", name="binary_output"),
        ],
        name="single_layer_perceptron",
    )
    model.compile(
        optimizer=optimizer,
        loss="binary_crossentropy",
        metrics=["accuracy"],
    )
    return model


build_single_layer_model(OPTIMIZER_SPECS["SGD (batch 1)"]["factory"]()).summary()

## 4. Train all six optimizers

Each run records training time, train loss/accuracy, and test loss/accuracy.
The raw histories are kept for the loss-curve plot.

In [ ]:
histories = {}
trained_models = {}
results = []

for name, spec in OPTIMIZER_SPECS.items():
    print(f"Training {name} ...")
    model = build_single_layer_model(spec["factory"]())
    start_time = time.perf_counter()
    history = model.fit(
        X_train,
        y_train,
        epochs=EPOCHS,
        batch_size=spec["batch_size"],
        shuffle=True,
        verbose=0,
    )
    elapsed = time.perf_counter() - start_time

    train_loss, train_accuracy = model.evaluate(X_train, y_train, verbose=0)
    test_loss, test_accuracy = model.evaluate(X_test, y_test, verbose=0)

    histories[name] = history
    trained_models[name] = model
    results.append(
        {
            "Optimizer": name,
            "Batch size": spec["batch_size"],
            "Train loss": float(train_loss),
            "Train accuracy": float(train_accuracy),
            "Test loss": float(test_loss),
            "Test accuracy": float(test_accuracy),
            "Seconds": float(elapsed),
        }
    )

results_df = pd.DataFrame(results)
print()
print(results_df.to_string(index=False, float_format=lambda value: f"{value:.4f}"))
results_df.to_csv(OUTPUT_DIR / "exp03_optimizer_results.csv", index=False)

## 5. Training-loss curves

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
for name in OPTIMIZER_SPECS:
    ax.plot(histories[name].history["loss"], label=name, linewidth=1.8)
ax.set_yscale("log")
ax.set_xlabel("Epoch")
ax.set_ylabel("Binary cross-entropy (log scale)")
ax.set_title("Loss by optimizer")
ax.grid(alpha=0.3)
ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "exp03_optimizer_losses.png", dpi=150, bbox_inches="tight")
plt.show()

## 6. Decision boundaries

In [ ]:
def plot_decision_boundary(ax, model, features, targets, title):
    x_min, x_max = features[:, 0].min() - 0.5, features[:, 0].max() + 0.5
    y_min, y_max = features[:, 1].min() - 0.5, features[:, 1].max() + 0.5
    xx, yy = np.meshgrid(
        np.linspace(x_min, x_max, 200),
        np.linspace(y_min, y_max, 200),
    )
    grid = np.c_[xx.ravel(), yy.ravel()].astype(np.float32)
    probabilities = model.predict(grid, verbose=0).reshape(xx.shape)

    ax.contourf(
        xx,
        yy,
        probabilities,
        levels=np.linspace(0.0, 1.0, 21),
        cmap="RdBu_r",
        alpha=0.75,
    )
    ax.contour(xx, yy, probabilities, levels=[0.5], colors="k", linewidths=1.5)
    ax.scatter(
        features[targets.ravel() == 0, 0],
        features[targets.ravel() == 0, 1],
        s=18,
        facecolor="tab:red",
        edgecolor="k",
        linewidth=0.3,
        label="Class 0",
    )
    ax.scatter(
        features[targets.ravel() == 1, 0],
        features[targets.ravel() == 1, 1],
        s=18,
        facecolor="tab:blue",
        edgecolor="k",
        linewidth=0.3,
        label="Class 1",
    )
    ax.set_title(title)
    ax.set_xlabel("x0")
    ax.set_ylabel("x1")
    ax.grid(alpha=0.2)


fig, axes = plt.subplots(2, 3, figsize=(14, 8))
for ax, (name, model) in zip(axes.ravel(), trained_models.items()):
    plot_decision_boundary(ax, model, X_train, y_train, name)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "exp03_decision_boundaries.png", dpi=150, bbox_inches="tight")
plt.show()

## 7. Test accuracy comparison

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
colors = ["tab:blue", "tab:orange", "tab:green", "tab:red", "tab:purple", "tab:brown"]
bars = ax.bar(results_df["Optimizer"], results_df["Test accuracy"], color=colors)
ax.set_ylim(0.90, 1.005)
ax.set_ylabel("Test accuracy")
ax.set_title("Single-layer network: test accuracy by optimizer")
ax.tick_params(axis="x", rotation=20)
ax.grid(axis="y", alpha=0.3)
for bar, value in zip(bars, results_df["Test accuracy"]):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.002,
        f"{value:.4f}",
        ha="center",
        va="bottom",
        fontsize=8,
    )
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "exp03_accuracy_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

## 8. Inference with the best model

In [ ]:
best_index = int(results_df["Test accuracy"].idxmax())
best_row = results_df.loc[best_index]
best_model = trained_models[best_row["Optimizer"]]

print(f"Best optimizer by test accuracy: {best_row['Optimizer']}")
print(f"Test accuracy: {best_row['Test accuracy']:.4f}")
print(f"Test loss    : {best_row['Test loss']:.4f}")
print()

test_probabilities = best_model.predict(X_test, verbose=0).ravel()
test_predictions = (test_probabilities >= 0.5).astype(int)
print(
    classification_report(
        y_test.ravel().astype(int),
        test_predictions,
        target_names=["class 0", "class 1"],
        digits=4,
        zero_division=0,
    )
)


def predict_class(x0, x1):
    sample = np.array([[x0, x1]], dtype=np.float32)
    probability = float(best_model.predict(sample, verbose=0)[0, 0])
    return int(probability >= 0.5), probability


for x0, x1 in [(-1.0, -1.0), (1.0, 1.0), (0.0, 0.0), (2.0, -1.0)]:
    label, probability = predict_class(x0, x1)
    print(f"x=({x0:+.1f}, {x1:+.1f}) -> class {label} (P(class 1) = {probability:.4f})")

## 9. Save the best model

In [ ]:
best_model_path = OUTPUT_DIR / "exp03_best_single_layer.keras"
best_model.save(best_model_path)
print(f"Saved best model to: {best_model_path.resolve()}")

reloaded = tf.keras.models.load_model(best_model_path)
reloaded_probabilities = reloaded.predict(X_test, verbose=0).ravel()
reloaded_predictions = (reloaded_probabilities >= 0.5).astype(int)
assert np.array_equal(reloaded_predictions, test_predictions)
print("Reload check passed: predictions are identical.")

## 10. Conclusion

All six optimizers train the same single sigmoid neuron to high accuracy on
this linearly separable dataset. The experiment shows that optimizer choice
affects the path and stability of training more than the final ceiling on an
easy problem. Momentum, Nesterov, and Adam tend to be more stable than plain
SGD; Adagrad can slow down as its accumulated squared gradients grow.

The original lab sheet records approximately **99.75% accuracy** for all six
variants, with losses between 0.01 and 0.05. Exact values in a rerun depend on
the TensorFlow version and the order of mini-batches.

**Artifacts created at runtime (not committed):**

- `outputs/exp03_optimizer_results.csv`
- `outputs/exp03_best_single_layer.keras`
- `outputs/exp03_dataset.png`
- `outputs/exp03_optimizer_losses.png`
- `outputs/exp03_decision_boundaries.png`
- `outputs/exp03_accuracy_comparison.png`

**Execution note:** the notebook was syntax-checked during repository
preparation; the full six-optimizer training run was not performed here.